# D02 — Per-perturbation severity references (optional data preparation)

**Optional notebook.** Builds the per-perturbation severity reference (leverage score) for each cell type by aggregating the guide-level rows of the Replogle 2022 supplementary tables to the gene level. The resulting CSVs are required only for full CPA/GEARS retraining (Phase 3 engineering layer) and for D03 (holdout-spec generation); they are **not** required for manuscript reproduction.

The default public reproducibility path bypasses severity-reference preparation entirely:

```bash
./run_all.sh --figures
```

which reproduces every manuscript figure, table, and Appendix B from the shipped evaluation CSVs in `precomputed/` in ~2 minutes.

This notebook is included for provenance: it documents the supplement-to-reference contract used by the author (see `TRAINING_PROVENANCE.md`).

**Manuscript reference:** Methods §2.1 (severity reference paragraph).

**Inputs (if executed):** Replogle Table S2 from the *Cell* paper, with Tab A (K562) and Tab C (RPE1) exported as CSVs:

1. Go to [doi.org/10.1016/j.cell.2022.05.013](https://doi.org/10.1016/j.cell.2022.05.013).
2. Scroll to Supplemental Information and download Table S2 (Excel).
3. Export Tab A as `data/replogle_supplement/TabA_K562_day8_summary.csv`.
4. Export Tab C as `data/replogle_supplement/TabC_RPE1_summary.csv`.

**Outputs (if executed):**
- `data/severity_refs/replogle_K562_severity.csv`
- `data/severity_refs/replogle_RPE1_severity.csv`


In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from perturb_style import DATA_DIR

SUPPLEMENT_DIR = DATA_DIR / "replogle_supplement"
OUT_DIR = DATA_DIR / "severity_refs"
OUT_DIR.mkdir(exist_ok=True, parents=True)

INPUTS = {
    "K562": SUPPLEMENT_DIR / "TabA_K562_day8_summary.csv",
    "RPE1": SUPPLEMENT_DIR / "TabC_RPE1_summary.csv",
}
OUTPUTS = {
    "K562": OUT_DIR / "replogle_K562_severity.csv",
    "RPE1": OUT_DIR / "replogle_RPE1_severity.csv",
}


## Check that the supplementary CSVs are present

In [ ]:
missing = [name for name, path in INPUTS.items() if not path.exists()]
if missing:
    print(f"Missing Replogle supplementary CSV(s): {missing}")
    print()
    print(f"Expected location: {SUPPLEMENT_DIR}/")
    for name, path in INPUTS.items():
        marker = "OK" if path.exists() else "MISSING"
        print(f"  [{marker}] {path.name}")
    print()
    print("Manual download from doi.org/10.1016/j.cell.2022.05.013, Table S2.")
    print("See this notebook's top markdown cell for full instructions.")
    print()
    print("If you only need to reproduce the figures and tables, run:")
    print("  ./run_all.sh --figures")
else:
    print(f"Both supplementary CSVs found in {SUPPLEMENT_DIR}/")


## Aggregate guide-level to gene-level

The Replogle supplementary tables provide leverage scores at the guide level; some genes are targeted by multiple guides. Aggregate by gene symbol, taking the mean of leverage score, percent knockdown, and DEG count across guides (after excluding control rows).

In [ ]:
if not missing:
    GUIDE_COL_CANDIDATES = ["guide", "guide_id", "sgRNA_id", "Guide"]
    GENE_COL_CANDIDATES = ["gene", "gene_symbol", "perturbation_target", "Gene"]
    LEV_COL_CANDIDATES = ["leverage_score", "leverage", "Leverage"]
    KD_COL_CANDIDATES = ["percent_knockdown", "knockdown_pct", "kd_pct", "PercentKnockdown"]
    DEG_COL_CANDIDATES = ["deg_count", "n_deg", "DEGs", "deg"]

    def find_col(df, candidates):
        for c in candidates:
            if c in df.columns:
                return c
        return None

    def aggregate(path: Path, cell_type: str) -> pd.DataFrame:
        df = pd.read_csv(path)
        print(f"{cell_type}: read {len(df)} rows from {path.name}")

        gene_col = find_col(df, GENE_COL_CANDIDATES)
        lev_col = find_col(df, LEV_COL_CANDIDATES)
        kd_col = find_col(df, KD_COL_CANDIDATES)
        deg_col = find_col(df, DEG_COL_CANDIDATES)

        if gene_col is None or lev_col is None:
            raise ValueError(
                f"Could not find expected columns in {path.name}. "
                f"Found columns: {list(df.columns)}. "
                f"Expected at least gene-symbol and leverage-score columns."
            )

        print(f"  using columns: gene={gene_col}, leverage={lev_col}, "
              f"knockdown={kd_col}, DEG={deg_col}")

        # Exclude control rows (heuristic: target gene name starts with "non-target" / "ctrl" / "scramble")
        before = len(df)
        mask = df[gene_col].astype(str).str.contains(r"non[-_]?target|control|ctrl|scramble", case=False, regex=True, na=False)
        df = df[~mask].copy()
        print(f"  excluded {before - len(df)} control rows; kept {len(df)} target rows")

        # Aggregate by gene
        agg_dict = {lev_col: "mean"}
        if kd_col is not None:
            agg_dict[kd_col] = "mean"
        if deg_col is not None:
            agg_dict[deg_col] = "mean"

        gene_df = df.groupby(gene_col, as_index=False).agg(agg_dict)
        gene_df = gene_df.rename(columns={
            gene_col: "perturbation_target",
            lev_col: "leverage_score",
            kd_col: "knockdown_pct" if kd_col else None,
            deg_col: "deg_count" if deg_col else None,
        })
        gene_df = gene_df.loc[:, [c for c in gene_df.columns if c is not None]]
        gene_df["n_source_rows"] = df.groupby(gene_col).size().values

        print(f"  aggregated to {len(gene_df)} gene-level rows")
        return gene_df

    for cell_type, in_path in INPUTS.items():
        out_path = OUTPUTS[cell_type]
        gene_df = aggregate(in_path, cell_type)
        gene_df.to_csv(out_path, index=False)
        print(f"  wrote {out_path}")
        print()
